## CASH Notebook

## Celestial Chase - LVL 1: The Teal Beacon

You've just woken up from cryo-sleep. No memory. No crew. Just you, a dying sun, and a faint signal pulsing from the sky.

One star glows different from the rest. Not white. Not grey. **Teal.**

Find it. Its position holds part of the key. But position alone is not enough - you must also know how much of its face is lit by the sun.

---

**The encoded signal:** `lpoo`

**Your task:**
1. Display the image and find the **cyan pixel** - it is the only pixel where `B == 255` and `G == 255` and `R == 0`
2. Read its `(x, y)` coordinates
3. Use `ephem` to compute **Venus's phase** (`int(venus.phase)`) on `2025/6/21 12:00:00` UTC from Zurich (lat=`47.3769`, lon=`8.5417`)
4. Build the key: `key = x * y + int(venus.phase)`
5. Build the permutation: `random.Random(key).shuffle(list(range(len(encoded))))`
6. Reverse the transposition to decode: `decoded[i] = encoded[perm[i]]`


In [ ]:
import base64, cv2, numpy as np
from IPython.display import display, Image as IPImage

# Starfield image embedded directly in this notebook
img_b64 = "iVBORw0KGgoAAAANSUhEUgAAAZAAAAGQCAIAAAAP3aGbAAAQvElEQVR4Ae3Bd3DWBZ7H8c/3v5sb93bd9XZtIDZQVhRkUcQGoUqLQGgSpIOhmCA99BY6JAYIRSBIkBbA0KQGsCDKIgguCjYE2+65697tzc39l5vxZmdkIPokecrv+/zer5cJAJwwAUhGg57pufyVdUouJkTJ95e+ur76LQIQMyYAUlHBivSMgUKwmcqxr3h7y7QOAoDAMCEa0lM7FpVsE+DTvuLtLdM6KPBMAOCECU50a9N+4+4dQqL9a1nZ/5gJiWDClaaMGD1lwVwBCB4TADhhAiri4tlzNerUFpAIJiAwsvoNzF21QvjBjvUb2/foJvyICQCcMAFAzBw/UNqweYqixAQATpgAwAmTf0Oe7bPk5TUC/ik1pXlJ6QEh6ZgAxNfiWXOHjhstVJwJAJwwAYATJgD+tXmiye7XDyvZmZLIhMwXZuQtFBJk6sgxk+fPEWLmrxcv/6ZGNYWYCQCcMAGAEyYAcMIEIBjGDc2ctThPKJ8JAJwwAYATJgBwwhSZVbn5/bKG6SrtGjfdeeSQACS1FQtyB47IUqKZgAB7ummLVw/tF/ADExCBsUOen73kRQEJZQKAxCkt2ZWS2laRMbnVoVnL7Qf3Kfa+v/TV9dVvEYBEMwGAEyYAcMKEynp7/6FHWjQVgHgxIfDe2LPv8dYthfK1a9x055FDQrIzAQiYvVu2tercUbiKKUh6tO+wfsd2AcC1mADE2Nlj79Rp9LBQZaaAmTZq7KR5sxUXw/sPWvTScgFwwgQATphCY8GU6SOmTBQAt0wA4IQJAJwwAaiymWPHj589U4gxExA8F8+eq1GntpLXtFFjJ82bLVSQCfjB3y9//atqNwsIMBOQ7DLSexUUrRX8M8GDj068d0+DBwWEmwkAnDABgBOmpPDB2+/e98hDgmc71m9s36ObgPKZkkXX1u027dkpAMnLBKB8x/YdbNSymRAMJgBwwgQATpiAnzNv0tRR0yYLSDQTADhhAoAKyuw7IG/1SsWdCQCcMAHB0KFZy+0H96kcF06erlm/rhBuJgBwwgQATpgAwAkTkCCvrnvl6Z7PCIiYCYjYFx98eNt99+rnDO7Ze+m6QgHRZkIoffvxZzfefYcS7YO3373vkYcERMYEIJTePXTkoaaN5YoJqLKZY8ePnz1TQIyZgHDLGTche9YMwQMTgKS2tXBdp949lRRMAOCECQCcMFXcq+teebrnMwLcGp0xdG7BYsEbU3x99/mlG26vLgCoOBMAOGECACdMAIKhd6cuhVs3C+UzAUA0dG3dbtOenYolExA9/1JW9r9mAmLDFG1jhzw/e8mLAoBoMwFAMPTr0n3V5g0qnwkAPBjef5AJAJwwAYATJgBwwgQAMVZUsCI9Y6D+6c+ffP67u25XxZmAYMtI71VQtFaAZEIsbV5V2KVfbwGIBhMAOGECACdMKMfWwnWdevcUgMAwAYmQPSwrJz9XQEWYAMAJE5JIfs6cYdljBMRXRnqvgqK1ij0TgCD58sMLt95bU7gWExB400aNnTRvthB6JgBwwuRWVr+BuatWCEBomADACVPVjM4YOrdgsZCMbiwr+9ZMQGCYAMAJEwAESWbfAXmrV+paTACQOF9+eOHWe2sqMiYAgdf2yZRdR0sVeqao+vi99+9+8AEBQAyYAMAJEwA4YQIAJ0wA4IQJAJwwOXei9GiDlCcF+JE7PSdrYrZQcSYAbvXq2Hntti0KDROQCBdOnq5Zv64QeoX5S3sPG6zImADACRMQDItnzR06brSA8pkQACsX5g14IVMAfpLpJx3YVtK8Y6oAhMCxfQcbtWymADMBgBMmRNuUEaOnLJgrROw/Pvvi3++4TT+yY/3G9j26CbiSCYmzZ1Nx665pAhAZE4CqKyuTmRBjJgBVVFam/2cmxJIJQNWVlclMiDETIG1bW9SxV7qAYDMhjpbPXzRo5HABqBQTADhhAgAnTFf56qOPb7nnbgEBdvHsuRp1agshYwICYM2LS/o8P0RItHFDM2ctzlOiHdhW0rxjqq5iApBo/bs+89KmV4SfY0LVfH/pq+ur3yIAsWcCACdMQPkmZo2YnrtAQDCYAMAJU/KaOXb8+NkzBSBZmDzrk9Z1TfEmAQgHEwA4YXLo2L6DjVo2U5B8c+HTm2reKQCxZAI8KMxf2nvYYFVEn7Sua4o3CUnEBMCh946++eCTjylkTAA8+Pr8JzfXukvhZgIAJ0yhV1K0ITW9u5LIsx3SXt5erNDY9NKarv37CCFgcqVPWtc1xZsEIJRMSCLHD5Q2bJ4iIEmZAMAJU2wM7J5+3XXXLVy5TMF2/o+nav2hngB4YAIAJ0zRs3vjljbdOgtICpOGj5y2aL4QJKYgOf/HU7X+UE/lKF7zclqfZwUgrEwA4IQJAJwwAYATJgBwwhQ9//nlN7+89SYBQGyYEqpd46Y7jxxSXHzw9rv3PfKQALhlgnRjWdm3ZoI3hflLew8bLISGCQCcMAGAE6ZQunzufLXateTfhZOna9avKyAcTADghAkAnDABgJTWsnXxvj0KNhMAxMvIQYPnL1+qyjLBlc2rCrv06y0glEwAIvCPr//8i5t/JySUCZVSUrQhNb27AMSRCYBP+7e+2qLT0woTEwA4YQIAJ0wA4IQpMqkpzUtKDwgAEscEAE6YgFDq1bHz2m1bFHclRRtS07sLlWICACdMAOCEKYl0a9N+4+4dApCkTKig4wdKGzZPEYC4M5Xv4/fev/vBBwQAwWACHMoelpWTnyuEjAkIsU9Onbmr3v2KryWz5w0ZO0qoOBMqYuPK1d0G9FXsrVyYd+bMmfzCVQLwTyYAcMIEAE6YXCkt2ZWS2lYAQskEAE6YAMAJU5V99v4HdzxwnwAgxkwA4IQJgXf6jWN1H2+kpDNjTPaEOTkCImYC3DpRerRBypNKCnkzZmVOGCf8JBMAOGECACdM8Kx9k2Y7Dh8UEA4mAFVTvObltD7PCrFnAgAnTADghAkAnDABgBMmAHDCBABOmADACVO0dW+bumFXiQAg2kxuDXqm5/JX1glAaJgQY4d37G7Svo1QQf/4+s+/uPl3An7EBABOmADACRMAOGECgGDo0b7D+h3bVT4TAATJ9WVl35vpWkwAECT5OXOGZY/RtZgAwAkTADhhAoB4ObJzT+N2rVVZJiChsodl5eTnCoiACQCcMAEB9qfjJ37fsIEq629ffPnr226Vf7nTc7ImZiv0TEhqe7dsa9W5o4CkYAIQsRcGPLdw5TIhQUwVkdVvYO6qFQKARDABgBMmAHDCBABOmADACRO8Gdyz99J1hQLCxwT84K29Bx5t1VxAgJni6409+x5v3VIAUHEmAHDCBCBx/uurb//tlhuFyJgAwAkTcKWvz39yc627BASPCQCcMAGIhi5Ptd382i4hlkwA4IQJAJwwAYATJkTb6IyhcwsWC0C0maLtk1Nn7qp3vwAkuz5pXdcUb1IcmQDACRMAOGECEHiLZ80dOm60Qs8E/MiRnXsat2stIJBMAPAjEzJfmJG3UIFkitjlc+er1a4lAEgQEwA4YQIAJ0wA4IQJQARGPTdk3rIlQkKZAEiZfQfkrV4pBJsJiLY/HT/x+4YNVEGv7977RJtWip4LJ0/XrF9XSCKmJDW0V9/Fa1cLgXf22Dt1Gj0sIAImBMPqvMV9M4cKQPlMAOCEqbLO//FUrT/UE2LmuR7PLlv/shAwryx/6ZlB/YVEMAXDZ+9/cMcD9wkAymcCEANvvrb/sadaCFFliqOZY8ePnz1TAFAppqSTnzNnWPYYAaisAd16rNy4XsFjcuL13XufaNNKAELMBABOmBA+w/sPWvTScuFa/vLpxd/eWUMIJBMAOGECACdMAOCECUg67Zs023H4oJB0TNGwdnFBr6EZE7NGTM9dIACIDRMAOGECACdMVdC3c7fVWzYqxgb37L10XaEAhJ4JoTewe/qKDUUCAs8EJFTejFmZE8YpiUx+YdTUhfOEGDABgBOmEOvQrOX2g/sEwAkTADhhAnClCydP16xfV4nz98tf/6razcJVTOHz2uatT3XpJADemADACRMARObbjz+78e47lDgmAAie+ZOnjZw6SVcyAeWbP3nayKmTBASDCVUwZ8LkMTOmCkBcmIAr/fXi5d/UqCYgeExAkjp55I36jR8XkogJAJwwAYATJgCouAPbSpp3TFV8mYCKGNqr7+K1qwUkggkAguHlJcueHfKcynF012smAHDCBABOmADACRNQQROzRkzPXSAg7kwA4IQJVfP2/kOPtGgqhNLciVNGT58ixIsJAJwwAaG0YkHuwBFZgismAHDCBABOmADACRMAxN13n1+64fbqqiATADhhQjLq1qb9xt07hNArKliRnjFQycIEAE6YAMAJEwA4YQIAJ0w+fXLqzF317he8eW3z1qe6dBJQKSbAsyM79zRu11oIBxMAOGFC9Hz3+aUbbq8uABW0ceXqbgP66ueYAP/eOXj44WZNBCdOlB5tkPKkKs6EYMhI71VQtFYAymcCACdMAOCEKQY+fPfkvQ/VFwBElQkAnDABye7AtpLmHVMF/0wAcJXubVM37CpRwJgAwAlTZY0YmLFgRYEAIF5MAOCECQCcMAFILgVzF2SMHqFkZAIAJ0xAlc0Ykz1hTo4QVTnjJmTPmiH8iAlIXn/74stf33arkCxMAOCEKV6OHyht2DxFCI33jr754JOPCYgeEyJw6vW36j3xqICwunDydM36dZVoJgBwwgQATpiAcPju80s33F5d8MwExF1m3wF5q1cKqCATklRpya6U1LYCkogJcfTOwcMPN2sixMaFk6dr1q8rJC9TWJ16/a16TzyqgJk0fOS0RfMF4FpMABBgHZq13H5wn35gCqpvLnx6U807hWg489bx+x9tKMA5U8Smjhwzef4cARHbs6m4ddc0ReCjE+/d0+BBAT/JBABOmAAEw+k3jtV9vJFQPlPgFcxdkDF6hAAE3uJZc4eOG60qe7ppi1cP7ddVTABCqX2TZjsOH1QMZPYdkLd6pWLAhOjp2LzVtgN7BSA2TADghAkAnDABgBMmAHDClDgTMl+YkbdQQHL572/+ct1NvxViwAQATpgABMOo54bMW7ZErvzl04u/vbOG4sUEAE6YAMAJEwA4YQqT3p26FG7dLAA+meDZjvUb2/foJiAcTADghAmIlzkTJo+ZMVVAZZkAwAkTADhhAmIgPbVjUck2AVFlAgAnTCjfXy9e/k2NagIQDCYAcMIEAE6YAMAJEwBEVX7OnGHZYxQDJgCIvdKSXSmpbVU1pqR25q3j9z/aUIhYRnqvgqK1AgLJBABOmABpz6bi1l3TFEtzJkweM2OqgCowAYATJgBwwhQCF8+eq1GntoCAmTR85LRF84WImQDACRMAn95/8+0HHntEgVdUsCI9Y6CiwQQATpgAwAkTADhhAgAnTADghMmP/Jw5w7LHCEBYmSqiU4untu5/TQCQCCYAcMIE4CrXl5V9byZc5auPPr7lnruVICZUTZ+0rmuKNwkxM2n4yGmL5guQTJG5fO58tdq1hMrav/XVFp2elkODe/Zeuq5QVTNt1NhJ82YLqBoTADhhAgAnTIiLfl26r9q8QUlq/uRpI6dOEhBjpjga9dyQecuWCMDPOfX6W/WeeFS4kgmIhrPH3qnT6GHBiUnDR05bNF/emADACRMAOGECACdMCJLn+/R/cc1LAmLmT8dP/L5hA/lkAkJj75ZtrTp3FNz6P5z3uSaepMPpAAAAAElFTkSuQmCC"

img_bytes = base64.b64decode(img_b64)
img_arr   = np.frombuffer(img_bytes, dtype=np.uint8)
img       = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)

cv2.imwrite('starfield.png', img)
display(IPImage('starfield.png'))
print("Image shape:", img.shape)


In [ ]:
import ephem, random

encoded  = "lpoo"
obs_date = "2025/6/21 12:00:00"
obs_lat  = "47.3769"
obs_lon  = "8.5417"

# TODO Step 1: Find the cyan pixel (B=255, G=255, R=0) in img
# Hint: use np.where or a loop
pixel_x, pixel_y = 0, 0  # replace with actual coordinates

# TODO Step 2: Compute Venus phase with ephem
obs = ephem.Observer()
obs.lat  = obs_lat
obs.lon  = obs_lon
obs.date = obs_date
venus = ephem.Venus()
venus.compute(obs)
phase = 0  # replace: int(venus.phase)

# TODO Step 3: Build key and permutation
key  = pixel_x * pixel_y + phase
perm = list(range(len(encoded)))
random.Random(key).shuffle(perm)

# TODO Step 4: Reverse the transposition
# Hint: if encoded[perm[i]] = original[i], then decoded[i] = encoded[perm[i]]? Think carefully.
def transpose_decode(encoded, perm):
    pass  # implement this

answer = transpose_decode(encoded, perm)
print(answer)
